## 55. How do you evaluate **retrieval quality**?

> **Answer:** “I evaluate retrieval separately from generation using a **golden dataset** containing queries and their expected relevant documents or chunks. I measure metrics such as **Precision@K, Recall@K, MRR, NDCG and Hit Rate**. I also compare retrieval performance before and after changes to chunking, embeddings, hybrid search, or reranking.”

### Key metrics

- **Precision@K** → How many retrieved chunks are actually relevant?
- **Recall@K** → How many of the relevant chunks were retrieved?
- **MRR** → How high is the first relevant result ranked?
- **NDCG@K** → How well are relevant results ranked, including relevance levels?
- **Hit Rate@K** → Did we retrieve at least one relevant chunk?

### Evaluation flow

```text id="w6hl0v"
Golden Dataset
     ↓
Query
     ↓
Retriever
     ↓
Top-K Results
     ↓
Compare with Ground Truth
     ↓
Precision / Recall / MRR / NDCG
```

### Example

```text id="1v5y3p"
Expected relevant chunks:
C2, C7

Retrieved Top-5:
C2, C4, C7, C9, C10

Precision@5 = 2/5
Recall@5 = 2/2
```

### RAG evaluation

After retrieval evaluation, I separately evaluate the generated answer using metrics such as:

```text
Retrieval
→ Precision@K / Recall@K / MRR / NDCG

Generation
→ Faithfulness / Answer Relevancy

Tools
→ RAGAS / DeepEval
```

**Interview one-liner:**

> **“I use a golden dataset and evaluate the retriever independently using Precision@K, Recall@K, MRR, NDCG and Hit Rate. This helps identify whether a poor RAG answer is caused by retrieval or by the LLM generation layer.”**

## 56. Explain **Precision@K**

> **Answer:** “Precision@K measures how many of the top K retrieved documents are actually relevant.”

### Formula

```text
Precision@K = Relevant Retrieved Documents / K
```

### Example

```text
Top-5 retrieved:
[D1, D2, D3, D4, D5]

Relevant:
D1, D3

Precision@5 = 2 / 5 = 0.40 = 40%
```

**Interview one-liner:**

> **“Precision@K tells me how relevant my retrieved Top-K results are.”**

---

## 57. Explain **Recall@K**

> **Answer:** “Recall@K measures how many of all the relevant documents were successfully retrieved in the top K results.”

### Formula

```text
Recall@K = Relevant Retrieved Documents / Total Relevant Documents
```

### Example

```text
Total relevant documents = 4

Retrieved Top-5:
[D1, D2, D3, D5, D8]

Relevant retrieved:
D1, D3

Recall@5 = 2 / 4 = 50%
```

**Interview one-liner:**

> **“Recall@K tells me how well my retriever finds the relevant information that exists in the knowledge base.”**

---

## 58. Precision vs Recall for RAG?

| | Precision | Recall |
|---|---|---|
| Measures | Relevance of retrieved results | Coverage of relevant results |
| Focus | Avoid irrelevant chunks | Avoid missing relevant chunks |
| High value means | Retrieved context is clean | Relevant information is found |
| RAG concern | Too much noise | Missing required context |

### Simple example

```text
Knowledge Base:
Relevant chunks = 5

Retriever returns Top-5:
3 relevant + 2 irrelevant

Precision@5 = 3/5 = 60%

Recall@5 = 3/5 = 60%
```

### In RAG

> **Precision is important because irrelevant context can confuse the LLM. Recall is important because missing the required information can cause an incomplete or hallucinated answer.**

A common production approach is:

```text
Initial Retrieval
     ↓
Higher Recall
     ↓
Top-20 / Top-50
     ↓
Reranking
     ↓
Higher Precision
     ↓
Top-5 / Top-10
     ↓
LLM
```

**Interview one-liner:**

> **“Recall helps me retrieve enough relevant information, while precision helps me keep the final context clean. In RAG, I typically optimize recall during initial retrieval and improve precision through reranking before sending context to the LLM.”**

## 59. What is **MRR (Mean Reciprocal Rank)**?

> **Answer:** “MRR measures how high the **first relevant result** appears in the retrieved ranking. It is particularly useful when getting one highly relevant document near the top is important.”

### Formula

```text
Reciprocal Rank = 1 / Rank of first relevant result

MRR = Average Reciprocal Rank across all queries
```

Example:

```text
Query 1 → first relevant result at rank 1 → 1/1 = 1.0
Query 2 → rank 2                     → 1/2 = 0.5
Query 3 → rank 4                     → 1/4 = 0.25

MRR = (1 + 0.5 + 0.25) / 3 = 0.583
```

**Interview one-liner:**

> **“MRR tells me how quickly the retriever places the first relevant result near the top.”**

---

## 60. What is **NDCG (Normalized Discounted Cumulative Gain)**?

> **Answer:** “NDCG evaluates the **quality and ordering of the top-K results**, including different levels of relevance. A highly relevant document at rank 1 contributes more than the same document at a lower rank.”

### Key idea

```text
Rank 1 → highest weight
Rank 2 → lower weight
Rank 3 → lower again
...
```

Unlike simple Precision@K, NDCG can handle graded relevance:

```text
3 → Highly relevant
2 → Relevant
1 → Partially relevant
0 → Irrelevant
```

```text
Query
 ↓
Ranked Results
 ↓
Relevance Scores
 ↓
Discount higher ranks
 ↓
Normalize
 ↓
NDCG@K
```

**Interview one-liner:**

> **“NDCG measures both relevance and ranking quality, giving more importance to highly relevant documents appearing near the top.”**

---

## 61. What is **Hit Rate**?

> **Answer:** “Hit Rate@K measures the percentage of queries for which **at least one relevant document appears in the top K results**.”

### Formula

```text
Hit Rate@K =
Queries with ≥1 relevant result in Top-K
-----------------------------------------
Total Queries
```

Example:

```text
100 queries

80 queries → at least one relevant chunk in Top-5

Hit Rate@5 = 80 / 100 = 80%
```

### RAG example

```text
Query
 ↓
Retriever
 ↓
Top-5
 ↓
Any relevant chunk?
 ├── Yes → HIT ✅
 └── No  → MISS ❌
```

**Interview one-liner:**

> **“Hit Rate@K tells me how often the retriever finds at least one relevant document within the top K results.”**

### Quick comparison

| Metric | Measures |
|---|---|
| **Precision@K** | How many retrieved results are relevant |
| **Recall@K** | How many relevant results were retrieved |
| **MRR** | Position of the first relevant result |
| **NDCG** | Relevance + ranking quality |
| **Hit Rate@K** | Whether at least one relevant result was found |

## 62. What is **Context Precision**?

> **Answer:** “Context Precision measures whether the **relevant retrieved chunks are ranked higher than irrelevant chunks**. It evaluates the quality of the retrieved context with respect to the question.”

```text
Query
 ↓
Retrieved Top-K
 ↓
Are relevant chunks ranked higher?
 ↓
Context Precision
```

**Example:**

```text
Top-5:
Relevant
Relevant
Irrelevant
Relevant
Irrelevant
```

Higher Context Precision means **less irrelevant context is being passed to the LLM**.

**Interview one-liner:**

> **“Context Precision measures how precisely the retriever ranks relevant information among the retrieved context.”**

---

## 63. What is **Context Recall**?

> **Answer:** “Context Recall measures whether the retrieved context contains **all the information required to answer the question**, compared with the available ground-truth information.”

```text
Ground Truth Information
          ↓
      Compare
          ↑
Retrieved Context
          ↓
    Context Recall
```

**Example:**

Suppose the expected answer requires 4 pieces of information:

```text
A, B, C, D
```

Retrieved context contains:

```text
A, B, C
```

Then the system missed **D**, so Context Recall is lower.

**Interview one-liner:**

> **“Context Recall measures how completely the retrieved context covers the information needed to answer the question.”**

---

## 64. What is **Faithfulness**?

> **Answer:** “Faithfulness measures whether the generated answer is **supported by the retrieved context** and does not introduce unsupported information.”

```text
Retrieved Context
       ↓
      LLM
       ↓
Generated Answer
       ↓
Are claims supported by context?
       ↓
  Faithfulness
```

### Example

**Context:**

```text
The policy provides 20 days of annual leave.
```

**Answer:**

```text
Employees receive 20 days of annual leave. ✅
```

High faithfulness.

But:

```text
Employees receive 30 days of annual leave. ❌
```

Low faithfulness because the answer is not supported by the retrieved context.

**Interview one-liner:**

> **“Faithfulness measures whether the generated answer is grounded in and supported by the retrieved context.”**

---

## 65. What is **Answer Relevancy**?

> **Answer:** “Answer Relevancy measures how well the generated answer actually addresses the **user's question**. An answer can be factually correct but still have low relevance if it doesn't directly address what the user asked.”

```text
User Question
      ↓
Generated Answer
      ↓
Does it address the question?
      ↓
Answer Relevancy
```

### Example

**Question:**

> “What is the company's leave policy?”

**Answer:**

> “The company was established in 1998 and has offices in Pune and Mumbai.”

The answer may be factually correct, but it is **not relevant** to the question.

**Interview one-liner:**

> **“Answer Relevancy measures whether the generated response directly and appropriately answers the user's question.”**

### Quick distinction

| Metric | What it evaluates |
|---|---|
| **Context Precision** | Are relevant retrieved chunks ranked highly? |
| **Context Recall** | Did we retrieve enough required information? |
| **Faithfulness** | Is the answer supported by the retrieved context? |
| **Answer Relevancy** | Does the answer address the user's question? |

### Easy memory trick

```text
Retrieval:
Precision → Relevant context ranked well
Recall    → Required context found

Generation:
Faithfulness → Answer supported by context
Relevancy    → Answer addresses the question
```

## 66. How would you use **RAGAS**?

> **Interview Answer:** “I use RAGAS to evaluate a RAG pipeline at both the **retrieval and generation levels**. I create a golden evaluation dataset containing questions, reference answers, and expected contexts where available. I run these questions through the RAG pipeline and evaluate metrics such as **Context Precision, Context Recall, Faithfulness, and Answer Relevancy**. I use the results to identify whether the problem is in retrieval, context quality, or LLM generation.”

---

### 1. Where RAGAS fits in the RAG architecture

```text
                RAG Pipeline
                    │
User Query ────────→│
                    ↓
              Retrieval
                    ↓
             Top-K Chunks
                    ↓
               Reranking
                    ↓
                 LLM
                    ↓
              Final Answer
                    │
                    ↓
                RAGAS
                    │
        ┌───────────┴────────────┐
        ↓                        ↓
   Retrieval Metrics       Generation Metrics
        ↓                        ↓
Context Precision          Faithfulness
Context Recall             Answer Relevancy
```

The key point is:

> **RAGAS is an evaluation layer around the RAG pipeline; it is not part of the runtime retrieval pipeline.**

---

# 2. First create an evaluation dataset

I normally create a **golden dataset** representing real production queries.

Example:

```python
dataset = [
    {
        "question": "What is the leave policy?",
        "answer": "Employees receive 20 days of annual leave.",
        "contexts": [
            "The company provides employees with 20 days of annual leave."
        ],
        "reference": "Employees receive 20 days of annual leave."
    }
]
```

In a real project, I would have hundreds or thousands of representative queries covering:

```text
Simple queries
Complex queries
Multi-hop queries
Ambiguous queries
No-answer queries
Different document types
Different tenants
Different business domains
```

---

# 3. Run the dataset through my RAG pipeline

For every question:

```text
Question
   ↓
Retriever
   ↓
Top-K
   ↓
Reranker
   ↓
Context
   ↓
LLM
   ↓
Answer
```

I capture:

```text
question
retrieved_contexts
generated_answer
reference_answer
```

For example:

```python
result = {
    "user_input": question,
    "retrieved_contexts": retrieved_chunks,
    "response": generated_answer,
    "reference": expected_answer
}
```

These become the inputs to RAGAS.

---

# 4. Context Precision

### What does it tell me?

> **“Are the relevant chunks ranked higher than irrelevant chunks?”**

Suppose:

```text
Top 5 retrieved chunks:

1. Relevant ✅
2. Relevant ✅
3. Irrelevant ❌
4. Irrelevant ❌
5. Relevant ✅
```

Context Precision tells me whether my retrieval ranking is putting useful information toward the top.

### If Context Precision is low

I investigate:

```text
Chunking
   ↓
Embedding model
   ↓
Search strategy
   ↓
Metadata filtering
   ↓
Top-K
   ↓
Reranking
```

For example, I might move from:

```text
Vector Search
```

to:

```text
Hybrid Search
      +
Reranking
```

---

# 5. Context Recall

### What does it tell me?

> **“Did I retrieve enough of the information required to answer the question?”**

Example:

Expected information:

```text
A + B + C + D
```

Retrieved:

```text
A + B + C
```

The retriever missed **D**.

Therefore Context Recall is low.

### If Context Recall is low

I investigate:

```text
Chunk size
Chunk overlap
Embedding model
Search K
Hybrid search
Metadata filters
Parent-child retrieval
Query transformation
```

For example:

```text
Top-K = 5
```

might become:

```text
Top-K = 20
      ↓
Reranker
      ↓
Top-5
```

This increases initial recall while keeping final context compact.

---

# 6. Faithfulness

### What does it tell me?

> **“Is the generated answer actually supported by the retrieved context?”**

Example:

**Retrieved context:**

```text
Company provides 20 days of annual leave.
```

**Generated answer:**

```text
Employees receive 20 days of annual leave.
```

High faithfulness.

But:

```text
Employees receive 30 days of annual leave.
```

Low faithfulness.

### If Faithfulness is low

I investigate:

```text
Poor retrieval
      ↓
Insufficient context
      ↓
LLM hallucination
      ↓
Poor grounding
```

Possible controls:

- Better retrieval
- Reranking
- Relevance threshold
- Grounded system prompt
- Structured output
- Citation validation
- "I don't know" fallback

---

# 7. Answer Relevancy

### What does it tell me?

> **“Does the generated answer actually address the user's question?”**

Question:

```text
What is the leave policy?
```

Answer:

```text
The company was founded in 1998 and has offices in Pune.
```

It may be factually correct, but it doesn't answer the question.

Therefore:

```text
Answer Relevancy = Low
```

### If Answer Relevancy is low

I investigate:

```text
Prompt
 ↓
Context construction
 ↓
LLM instructions
 ↓
Output format
```

---

# 8. The four metrics together

This is very important for interviews.

```text
                  RAGAS
                    │
       ┌────────────┴────────────┐
       │                         │
   Retrieval                  Generation
       │                         │
       ↓                         ↓
Context Precision          Faithfulness
Context Recall             Answer Relevancy
```

### Easy memory trick

```text
Context Precision
→ Is retrieved context precise?

Context Recall
→ Did I retrieve enough?

Faithfulness
→ Is my answer supported?

Answer Relevancy
→ Did I answer the question?
```

---

# 9. Simple RAGAS code

Conceptually, the evaluation looks like:

```python
from ragas import evaluate
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    ResponseRelevancy
)

result = evaluate(
    dataset,
    metrics=[
        ContextPrecision(),
        ContextRecall(),
        Faithfulness(),
        ResponseRelevancy()
    ]
)

print(result)
```

The exact RAGAS APIs can vary by version, so in a production implementation I pin the RAGAS version and use the corresponding metric/evaluator interfaces.

---

# 10. How I interpret the results

Suppose I get:

```text
Context Precision = 0.82
Context Recall    = 0.61
Faithfulness      = 0.91
Answer Relevancy  = 0.88
```

My interpretation:

```text
Precision = Good
Recall    = Weak
Faithfulness = Good
Relevancy   = Good
```

Therefore:

> **The generation layer is reasonably good, but retrieval is missing relevant information.**

I would focus on:

```text
Chunking
Embeddings
Hybrid Search
Top-K
Query Expansion
Parent-Child Retrieval
Reranking
```

rather than immediately changing the LLM.

---

# 11. Another example

Suppose:

```text
Context Precision = 0.45
Context Recall    = 0.85
Faithfulness      = 0.70
Answer Relevancy  = 0.90
```

Interpretation:

```text
Recall → Good
Precision → Poor
```

We are finding the required information, **but also retrieving lots of irrelevant information**.

I would focus on:

```text
Metadata filtering
Reranking
Top-K
Relevance threshold
Hybrid search
```

---

# 12. RAGAS vs traditional retrieval metrics

This is a common interview follow-up.

### Traditional retrieval evaluation

```text
Precision@K
Recall@K
MRR
NDCG
Hit Rate
```

These primarily evaluate the **retrieval ranking** against known ground truth.

### RAGAS

```text
Context Precision
Context Recall
Faithfulness
Answer Relevancy
```

These evaluate the **RAG pipeline more holistically**, including the relationship between retrieved context and generated answer.

So I use both.

```text
                  RAG Evaluation
                       │
          ┌────────────┴─────────────┐
          ↓                          ↓
   Retrieval Evaluation       End-to-End Evaluation
          ↓                          ↓
 Precision@K                  Faithfulness
 Recall@K                     Answer Relevancy
 MRR                          Context Precision
 NDCG                         Context Recall
 Hit Rate
```

---

# 13. RAGAS in a production CI/CD pipeline

This is a strong **production-system-design answer**.

I don't just run RAGAS once.

```text
Developer changes
      ↓
Chunking / Prompt / Embedding
      ↓
CI Pipeline
      ↓
Golden Dataset
      ↓
RAG Evaluation
      ↓
RAGAS
      ↓
Threshold Check
   ┌──┴────┐
 PASS     FAIL
  ↓         ↓
Deploy    Reject
```

For example:

```python
if faithfulness < 0.85:
    raise Exception("RAG evaluation failed")
```

Similarly, I can establish thresholds for:

```text
Context Precision
Context Recall
Faithfulness
Answer Relevancy
```

This prevents a new prompt, embedding model, chunking strategy, or reranker from silently degrading production quality.

---

# 14. How I would use RAGAS in your AutoShift project

For your **AutoShift** scenario, I would create a golden dataset such as:

```text
Question
Expected Client
Expected Location
Expected Qualification
Expected Intent
Expected Answer
Expected Context
```

Example:

```text
Question:
"Please schedule a Java resource in Pune tomorrow
from 9 AM to 6 PM."

Expected:
Client       → ABC
Location     → Pune
Qualification→ Java Developer
Intent       → CREATE_SHIFT
```

Then evaluate:

```text
Email
 ↓
Retrieval
 ↓
Qdrant
 ↓
Top-K
 ↓
Reranker
 ↓
Claude
 ↓
Structured JSON
 ↓
RAGAS / DeepEval
```

I would additionally measure **business-level metrics** because RAGAS alone doesn't fully evaluate whether the correct workforce action was executed:

```text
Intent Accuracy
Field Extraction Accuracy
Tool Selection Accuracy
API Success Rate
HIL Rate
End-to-End Workflow Accuracy
```

---

## Strong interview answer

> **“I use RAGAS as an evaluation layer for my RAG pipeline. I first build a golden dataset containing representative production queries, expected answers, and relevant contexts. I run those queries through the complete RAG pipeline and evaluate Context Precision and Context Recall for retrieval, and Faithfulness and Answer Relevancy for generation. If Context Recall is low, I tune chunking, embeddings, Top-K or retrieval strategy. If Precision is low, I tune filtering or reranking. If Faithfulness is low, I focus on grounding and hallucination controls. I integrate these evaluations into CI/CD with quality thresholds so changes to prompts, embeddings, chunking or models don't degrade production performance.”**

### One-line architecture

```text
Golden Dataset → RAG Pipeline → RAGAS → Metrics → Thresholds → Deploy / Reject
```

**Key distinction to remember for interviews:**

> **“RAGAS tells me where the RAG pipeline is failing; it doesn't replace retrieval metrics or business-level evaluation.”**

## 67. How would you use **DeepEval**?

> **Interview Answer:** “I use DeepEval as an automated evaluation framework for LLM and Agentic AI applications. For a RAG system, I create a golden dataset, execute the complete pipeline, and evaluate metrics such as **Answer Relevancy, Faithfulness, Contextual Precision, Contextual Recall, and Contextual Relevancy**. For agentic workflows, I additionally evaluate tool selection, task completion, and safety-related behavior. I integrate these evaluations into CI/CD so a model, prompt, retrieval, or agent change can be automatically validated before deployment.”

---

### 1. Where DeepEval fits

```text
                  Production AI
                       │
              ┌────────┴────────┐
              ↓                 ↓
             RAG             Agents
              │                 │
         Retrieval          Tool Calls
              ↓                 ↓
             LLM            Decisions
              └────────┬────────┘
                       ↓
                    DeepEval
                       ↓
                  Evaluation
                       ↓
                Pass / Fail Gate
```

DeepEval is primarily an **evaluation/testing layer**, not part of the runtime RAG execution path.

---

# 2. Start with a golden dataset

I create representative test cases from production scenarios.

```python
test_case = {
    "input": "What is the leave policy?",
    "actual_output": "Employees receive 20 days of annual leave.",
    "expected_output": "Employees receive 20 days of annual leave.",
    "retrieval_context": [
        "Employees receive 20 days of annual leave."
    ]
}
```

For a production system, I would include:

```text
Simple questions
Complex questions
Multi-hop questions
No-answer questions
Ambiguous questions
Edge cases
Domain-specific questions
Safety cases
```

---

# 3. RAG evaluation with DeepEval

The basic flow is:

```text
Golden Dataset
      ↓
Run RAG
      ↓
Capture
 ├── Input
 ├── Output
 └── Retrieved Context
      ↓
DeepEval Metrics
      ↓
Scores
      ↓
Threshold
      ↓
PASS / FAIL
```

Example structure:

```python
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric
)

test_case = LLMTestCase(
    input="What is the leave policy?",
    actual_output=answer,
    expected_output=reference_answer,
    retrieval_context=retrieved_context
)
```

Then define metrics:

```python
metrics = [
    AnswerRelevancyMetric(threshold=0.8),
    FaithfulnessMetric(threshold=0.8),
    ContextualPrecisionMetric(threshold=0.8),
    ContextualRecallMetric(threshold=0.8)
]
```

And evaluate:

```python
evaluate(
    test_cases=[test_case],
    metrics=metrics
)
```

---

# 4. Answer Relevancy

### Question

> Does the generated answer actually address the user's question?

```text
Question
   ↓
Generated Answer
   ↓
DeepEval
   ↓
Answer Relevancy
```

Example:

```text
Question:
"What is the leave policy?"

Answer:
"The company has offices in Pune and Mumbai."
```

The answer may be factually correct, but it doesn't answer the question.

Therefore:

```text
Answer Relevancy → LOW
```

---

# 5. Faithfulness

### Question

> Is the answer supported by the retrieved context?

Context:

```text
Company provides 20 days annual leave.
```

Answer:

```text
Employees receive 20 days annual leave.
```

```text
Faithfulness → HIGH
```

But:

```text
Employees receive 30 days annual leave.
```

```text
Faithfulness → LOW
```

This is particularly important for detecting **RAG hallucinations**.

---

# 6. Contextual Precision

### Question

> Are the relevant retrieved chunks ranked appropriately?

Example:

```text
Top 5:

1. Relevant
2. Relevant
3. Irrelevant
4. Irrelevant
5. Relevant
```

DeepEval helps identify whether the retrieval pipeline is returning useful context rather than flooding the LLM with irrelevant chunks.

If low:

```text
Check:
Chunking
Embeddings
Hybrid Search
Top-K
Reranking
Metadata filters
```

---

# 7. Contextual Recall

### Question

> Did the retrieved context contain the information required to answer the question?

Expected information:

```text
A + B + C + D
```

Retrieved:

```text
A + B + C
```

Information **D** is missing.

Therefore:

```text
Contextual Recall → LOW
```

I would investigate:

```text
Chunking
Embedding model
Top-K
Query transformation
Parent-child retrieval
Hybrid search
```

---

# 8. Contextual Relevancy

This evaluates whether the retrieved context is actually relevant to the input.

```text
Question
   ↓
Retrieved Context
   ↓
Is context relevant?
   ↓
Contextual Relevancy
```

This is useful for detecting:

```text
Too many irrelevant chunks
Poor retrieval
Poor metadata filtering
Incorrect semantic matches
```

---

# 9. DeepEval for Agentic AI

This is where DeepEval becomes particularly useful for your **Agentic AI interview preparation**.

For an agent:

```text
User
 ↓
Supervisor / Router
 ↓
Agent
 ↓
Tool Selection
 ↓
Tool Execution
 ↓
Final Answer
```

I can evaluate more than just the final answer.

For example:

```text
Tool Selection
Task Completion
Reasoning / Trajectory
Final Answer
Safety
```

### Example

User:

```text
"Find the latest medical research on diabetes."
```

Expected:

```text
Agent → PubMed Search
```

Actual:

```text
Agent → Web Search
```

The final response might look reasonable, but the **agent trajectory is incorrect**.

Therefore I evaluate the intermediate behavior as well.

---

# 10. Agent trajectory evaluation

For an agentic workflow I capture:

```python
trace = {
    "input": question,
    "steps": [
        "Router",
        "PubMed Agent",
        "PubMed Search",
        "Answer"
    ],
    "tools": [
        "pubmed_search"
    ],
    "output": answer
}
```

Then evaluate whether the execution path matches the expected behavior.

Conceptually:

```text
Expected:
Router → PubMed → Answer

Actual:
Router → Web → Answer

                ↓
             FAIL ❌
```

This is extremely useful for **tool-routing and multi-agent systems**.

---

# 11. DeepEval for AutoShift

For your AutoShift example, I would create test cases such as:

```text
Input:
"Please schedule a Java developer in Pune
tomorrow from 9 AM to 6 PM."

Expected:
Intent = CREATE_SHIFT
Location = Pune
Qualification = Java Developer
```

Then evaluate:

```text
Email Extraction
      ↓
RAG Retrieval
      ↓
Intent Detection
      ↓
Structured Extraction
      ↓
Tool Selection
      ↓
API Action
```

Metrics could include:

```text
Retrieval Quality
Intent Accuracy
Field Extraction Accuracy
Tool Selection
Task Completion
Faithfulness
Answer Relevancy
```

For an agentic system, I would complement generic LLM metrics with **business-specific assertions**.

---

# 12. Custom metrics

One major advantage is that I don't have to rely only on generic RAG metrics.

For example:

```python
expected_intent = "CREATE_SHIFT"
actual_intent = result["intent"]

assert actual_intent == expected_intent
```

Or:

```python
assert result["location"] == "Pune"
assert result["qualification"] == "Java Developer"
```

This gives me **domain-specific evaluation**.

---

# 13. CI/CD integration

This is a strong production interview point.

I would run DeepEval as part of CI/CD:

```text
Developer Changes
      ↓
Prompt / Model / RAG / Agent Change
      ↓
CI Pipeline
      ↓
Golden Dataset
      ↓
DeepEval
      ↓
Metrics
      ↓
Threshold Check
   ┌──┴───┐
 PASS   FAIL
  ↓       ↓
Deploy   Reject
```

Example:

```python
metric = FaithfulnessMetric(
    threshold=0.85
)
```

If the evaluation falls below the agreed threshold:

```text
Build → FAILED ❌
```

This prevents a new prompt or model from silently degrading the production system.

---

# 14. Debugging with DeepEval

Suppose:

```text
Answer Relevancy  = 0.92
Faithfulness      = 0.91
Context Precision = 0.52
Context Recall    = 0.87
```

I would conclude:

```text
Generation → Good
Recall      → Good
Precision   → Poor
```

So I would **not immediately change the LLM**.

I would investigate:

```text
Metadata filters
Reranking
Top-K
Hybrid Search
Chunking
```

Another scenario:

```text
Context Precision = 0.90
Context Recall    = 0.91
Faithfulness      = 0.55
```

Now retrieval looks good, but generation is poorly grounded.

I would investigate:

```text
Prompt grounding
LLM behavior
Context construction
Output validation
Hallucination controls
```

---

# 15. DeepEval vs RAGAS

A very good interview follow-up is:

> **“Why DeepEval if you already use RAGAS?”**

My answer:

> **“RAGAS is particularly useful for evaluating RAG retrieval and generation quality, while DeepEval provides a broader LLM evaluation and testing framework that I can extend to RAG, agents, tool usage, and custom business metrics. In production, I can use both depending on the evaluation requirements.”**

### Comparison

| RAGAS | DeepEval |
|---|---|
| Strong RAG focus | Broader LLM evaluation |
| Context Precision | Contextual Precision |
| Context Recall | Contextual Recall |
| Faithfulness | Faithfulness |
| Answer Relevancy | Answer Relevancy |
| RAG evaluation | RAG + Agent evaluation |
| Retrieval-centric | Test/CI-centric |
| Golden datasets | Test cases + custom metrics |

---

# 16. RAGAS + DeepEval together

For a production RAG system, I can use:

```text
                 Evaluation
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
        RAGAS                 DeepEval
          ↓                     ↓
 Retrieval + RAG           LLM + Agent
          ↓                     ↓
 Precision/Recall          Faithfulness
 Context metrics           Relevancy
                            Tool behavior
                            Custom metrics
          └──────────┬──────────┘
                     ↓
               Quality Gate
                     ↓
              Deploy / Reject
```

---

# 17. Production evaluation architecture

```text
                  Golden Dataset
                       ↓
                Evaluation Runner
                       ↓
              ┌────────┴─────────┐
              ↓                  ↓
             RAG              Agent
              ↓                  ↓
          Retrieval           Tools
              ↓                  ↓
             LLM              LLM
              └────────┬─────────┘
                       ↓
                  DeepEval
                       ↓
              Metrics + Scores
                       ↓
              Quality Threshold
                       ↓
               ┌───────┴───────┐
               ↓               ↓
             PASS             FAIL
               ↓               ↓
            Deploy        Investigate
```

---

## Strong interview answer

> **“I use DeepEval as an automated evaluation and regression-testing layer for my LLM and Agentic AI systems. I create a golden dataset from representative production scenarios, execute the complete RAG or agent workflow, and capture inputs, outputs, retrieved context and agent trajectories. For RAG, I evaluate metrics such as contextual precision, contextual recall, faithfulness and answer relevancy. For agents, I additionally evaluate task completion, tool selection, trajectory and custom business assertions. I integrate these evaluations into CI/CD with predefined thresholds, so any degradation caused by changes to the model, prompt, embedding, retrieval strategy or agent workflow can block deployment.”**

### Easy interview memory

```text
RAGAS
→ "Is my RAG pipeline retrieving and answering well?"

DeepEval
→ "Can I systematically test and regression-test
   my LLM/RAG/Agent system?"
```

**Most important distinction:** Don't say *“DeepEval prevents hallucinations.”* Say **“DeepEval detects and measures quality problems such as unfaithful or irrelevant responses; I then use guardrails, retrieval controls, prompting, validation, and fallback logic to prevent them at runtime.”**

## 68. How do you create a **Golden Dataset**?

> **Interview Answer:** “I create a golden dataset from representative production queries and manually validated expected outcomes. I include normal cases, edge cases, ambiguous queries, no-answer cases, and failure scenarios. Each test case contains the input, expected answer or ground truth, and where applicable, the expected relevant context. I then use this dataset for RAGAS, DeepEval, regression testing, and CI/CD quality gates.”

### 1. Start from real production queries

I collect queries from:

```text
Production Logs
      +
Business Examples
      +
Domain Experts
      +
Known Failure Cases
      ↓
Candidate Dataset
```

For example, in RAG:

```text
Question:
"What is the leave policy?"

Expected Answer:
"Employees receive 20 days of annual leave."

Relevant Context:
"Employees receive 20 days of annual leave."
```

---

### 2. Cover different query categories

I don't create only simple questions.

```text
Golden Dataset
├── Simple queries
├── Complex queries
├── Multi-hop queries
├── Ambiguous queries
├── No-answer queries
├── Edge cases
├── Domain-specific queries
└── Previously failed queries
```

For example:

```text
"What is the leave policy?"          → Simple
"Compare leave policies for 2025/26" → Complex
"Who is eligible and how many days?" → Multi-hop
"Is this applicable to contractors?" → Edge case
"What is our policy on X?"           → No-answer
```

---

### 3. Define ground truth

For each query, domain experts validate:

```python
{
    "question": "...",
    "reference_answer": "...",
    "relevant_context": [...],
    "metadata": {...}
}
```

For agentic systems, I can additionally define:

```python
{
    "expected_intent": "CREATE_SHIFT",
    "expected_tool": "create_shift",
    "expected_parameters": {
        "location": "Pune",
        "qualification": "Java Developer"
    }
}
```

---

### 4. Validate with domain experts

This is critical.

```text
Candidate Dataset
       ↓
Domain Expert Review
       ↓
Correct?
 ┌─────┴─────┐
No          Yes
 ↓            ↓
Fix         Golden
```

The golden dataset should represent **business-approved expected behavior**, not simply LLM-generated answers.

---

### 5. Split the dataset

I normally maintain:

```text
Golden Dataset
├── Development set
├── Validation set
└── Test set
```

The **test set should remain stable and protected** so developers don't continuously tune the system against the same test cases.

---

### 6. Use it for regression testing

```text
Change
 ↓
Prompt / LLM / Embedding / Chunking / Reranker
 ↓
Run Golden Dataset
 ↓
RAGAS + DeepEval
 ↓
Compare with Baseline
 ↓
PASS / FAIL
```

Example quality gate:

```python
if faithfulness < 0.85:
    raise Exception("Evaluation failed")
```

---

### Strong interview answer

> **“I create a golden dataset from real production queries, business requirements, domain-expert annotations, and historical failure cases. I cover normal, complex, ambiguous, edge and no-answer scenarios. Each case contains the input and validated expected output, and for RAG also the relevant context. For agents, I can additionally capture expected intent, tool and parameters. I version and protect the test set and use it with RAGAS and DeepEval as a regression suite and CI/CD quality gate.”**

### Easy memory

```text
Real Queries
     ↓
Expert Annotation
     ↓
Ground Truth
     ↓
Golden Dataset
     ↓
RAGAS / DeepEval
     ↓
Regression Testing
```

## 69. How do you evaluate an **agent workflow end-to-end**?

> **Interview Answer:** “I evaluate the agent at multiple levels—not only the final answer. I validate the **input, routing decision, tool selection, tool parameters, execution result, final response, latency, cost, and safety**. I use a golden dataset with RAGAS/DeepEval for automated evaluation and business-specific assertions for workflow correctness.”

### End-to-end evaluation

```text
Golden Dataset
      ↓
Agent Workflow
      ↓
┌──────────────────────────┐
│ 1. Intent / Routing      │
│ 2. Agent Decision        │
│ 3. Tool Selection        │
│ 4. Tool Parameters       │
│ 5. Tool Execution        │
│ 6. State Transitions     │
│ 7. Final Answer          │
└─────────────┬────────────┘
              ↓
        Evaluation
              ↓
      PASS / FAIL / SCORE
```

### What I measure

**1. Functional correctness**

```text
Intent Accuracy
Tool Selection Accuracy
Parameter Accuracy
Task Completion Rate
Final Answer Accuracy
```

**2. RAG quality**

```text
Context Precision
Context Recall
Faithfulness
Answer Relevancy
```

**3. Agent behavior**

```text
Correct routing
Correct tool usage
No unnecessary tool calls
No infinite loops
Correct termination
Correct state transitions
```

**4. Production metrics**

```text
Latency
Token usage
Cost
Error rate
Retry count
HIL rate
```

**5. Safety**

```text
Prompt injection
Unauthorized tool calls
PII leakage
Unsafe actions
Policy violations
```

### Example — AutoShift

```text
Email
 ↓
Intent = CREATE_SHIFT       ✓
 ↓
Tool = create_shift         ✓
 ↓
Location = Pune             ✓
Qualification = Java        ✓
 ↓
API Success                 ✓
 ↓
Correct final response      ✓
```

If the final answer is correct but the agent selected the **wrong tool and happened to get the correct result**, I would still mark the workflow as a failure because the **trajectory was incorrect**.

### Production evaluation architecture

```text
Golden Dataset
      ↓
Agent Execution
      ↓
LangSmith Trace
      ↓
DeepEval / RAGAS
      +
Business Assertions
      ↓
Quality Thresholds
      ↓
CI/CD Gate
```

**Interview one-liner:**

> **“I evaluate an agent end-to-end by measuring both the final outcome and the execution trajectory—intent, routing, tool selection, parameters, state transitions, task completion, safety, latency and cost—and combine DeepEval/RAGAS with domain-specific business assertions for the final quality gate.”**

## 70. How would you detect **RAG quality degradation after deployment**?

> **Interview Answer:** “I would establish a **production quality baseline** before deployment and continuously monitor retrieval and generation metrics against that baseline. I would combine online monitoring, user feedback, sampled production queries, and periodic RAGAS/DeepEval evaluation. If metrics cross predefined thresholds, I would trigger an alert and investigate whether the degradation is coming from retrieval, the LLM, data freshness, or infrastructure.”

### Production monitoring

```text
Production Queries
       ↓
   Sample Queries
       ↓
┌──────────────────────┐
│ Retrieval Metrics    │
│ Precision / Recall   │
│ Hit Rate / MRR       │
└──────────┬───────────┘
           ↓
┌──────────────────────┐
│ RAG Metrics          │
│ Faithfulness         │
│ Context Precision    │
│ Context Recall       │
│ Answer Relevancy     │
└──────────┬───────────┘
           ↓
     Compare Baseline
           ↓
     Threshold Breach?
       ┌────┴────┐
      No        Yes
       ↓          ↓
    Continue    Alert
                  ↓
             Root Cause
```

### What I monitor

**Retrieval degradation**
- Recall@K
- Precision@K
- MRR / NDCG
- Hit Rate
- Context Precision
- Context Recall

**Generation degradation**
- Faithfulness
- Answer Relevancy
- "I don't know" rate
- Citation correctness

**Operational signals**
- LLM latency
- Retrieval latency
- Error rate
- Token/cost increase
- HIL/escalation rate

### Root-cause analysis

```text
Quality ↓
   ↓
Retrieval ↓ ? → Chunking / Embedding / Search / Reranker
   ↓
Generation ↓ ? → Prompt / Model / Context
   ↓
Data ↓ ? → Stale / Missing / Deleted documents
```

### Production feedback loop

```text
Production Traffic
       ↓
Monitoring + Sampling
       ↓
RAGAS / DeepEval
       ↓
Baseline Comparison
       ↓
Alert
       ↓
Root Cause Analysis
       ↓
Fix
       ↓
Regression Evaluation
       ↓
Deploy
```

**Interview one-liner:**

> **“I detect RAG degradation by continuously comparing production retrieval, generation, operational, and business metrics against a validated baseline, using RAGAS/DeepEval on sampled queries. I alert on threshold breaches and then isolate whether the issue is retrieval, generation, data freshness, or infrastructure.”**